# Espectro observado × limiar da análise paralela

Este notebook compara, para os nove cenários da campanha FULL, os valores singulares observados $\sigma_r(\mathbf{E}_G)$ com os limiares $\tau_r$ da análise paralela. Cada limiar corresponde ao percentil 95 de 2.000 espectros Monte Carlo gerados sob o modelo de ruído independente usado no benchmark.

Como cada cenário foi avaliado em dez sementes, as linhas mostram as medianas e as faixas sombreadas mostram os intervalos interquartis. Uma direção é retida quando $\sigma_r(\mathbf E_G)>\tau_r$. A anotação $d$ resume a mediana e a amplitude da dimensão efetiva observada nas dez repetições.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

def project_root(start=Path.cwd()):
    path = start.resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'configs' / 'full.json').exists():
            return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada.')

ROOT = project_root()
SOURCE = ROOT / 'results' / 'synthetic' / 'tables' / 'full_parallel_analysis.csv'
OUT_DIR = ROOT / 'results' / 'synthetic' / 'figures' / 'spectral_analysis'
OUT_DIR.mkdir(parents=True, exist_ok=True)

M_VALUES = (4, 6, 12)
CORRELATION_LEVELS = ('low', 'medium', 'high')
EXPECTED_SCENARIOS = [f'm{m}_{level}' for m in M_VALUES for level in CORRELATION_LEVELS]
OBSERVED_COLOR = '#2166AC'
THRESHOLD_COLOR = '#B2182B'
FIGURE_WIDTH_CM = 16.0
FIGURE_HEIGHT_CM = 18.5
CM_TO_INCH = 1 / 2.54
OUTPUT_STEM = 'nove_cenarios_espectro_observado_vs_analise_paralela'

plt.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 9.5,
    'axes.titlesize': 10.5,
    'axes.labelsize': 9.5,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 7.5,
    'legend.fontsize': 9.0,
    'savefig.facecolor': 'white',
    'axes.facecolor': 'white',
})

print('Fonte:', SOURCE)
print('Saída:', OUT_DIR)

In [ ]:
if not SOURCE.exists():
    raise FileNotFoundError(f'Tabela da campanha FULL ausente: {SOURCE}')

raw = pd.read_csv(SOURCE)
required = {
    'scenario', 'seed', 'singular_index', 'singular_value', 'noise_p95',
    'retained_d', 'n_mc', 'mc_seed', 'mode', 'floor', 'ceiling',
}
missing = required - set(raw.columns)
if missing:
    raise ValueError(f'Colunas ausentes: {sorted(missing)}')

raw = raw[raw['scenario'].isin(EXPECTED_SCENARIOS)].copy()
assert set(raw['scenario']) == set(EXPECTED_SCENARIOS)
assert raw[['singular_value', 'noise_p95']].notna().all().all()
assert (raw[['singular_value', 'noise_p95']] >= 0).all().all()
assert raw[['scenario', 'seed', 'singular_index']].duplicated().sum() == 0
assert set(raw['n_mc']) == {2000} and set(raw['mc_seed']) == {777}
assert set(raw['mode']) == {'independent_legacy'}

for scenario in EXPECTED_SCENARIOS:
    scenario_rows = raw[raw['scenario'] == scenario]
    m = int(scenario.split('_')[0][1:])
    assert scenario_rows['seed'].nunique() == 10
    assert set(scenario_rows['singular_index']) == set(range(1, m))
    assert scenario_rows.groupby('seed').size().eq(m - 1).all()
    recomputed = scenario_rows.groupby('seed').apply(
        lambda frame: max(1, int((frame['singular_value'] > frame['noise_p95']).sum())),
        include_groups=False,
    )
    reported = scenario_rows.groupby('seed')['retained_d'].first().astype(int)
    assert recomputed.index.equals(reported.index)
    assert np.array_equal(recomputed.to_numpy(), reported.to_numpy())

summary = (
    raw.groupby(['scenario', 'singular_index'], as_index=False)
       .agg(
           observed_median=('singular_value', 'median'),
           observed_q1=('singular_value', lambda values: values.quantile(0.25)),
           observed_q3=('singular_value', lambda values: values.quantile(0.75)),
           threshold_median=('noise_p95', 'median'),
           threshold_q1=('noise_p95', lambda values: values.quantile(0.25)),
           threshold_q3=('noise_p95', lambda values: values.quantile(0.75)),
           seeds=('seed', 'nunique'),
       )
)

dimension_summary = (
    raw[['scenario', 'seed', 'retained_d']].drop_duplicates()
       .groupby('scenario')['retained_d']
       .agg(d_median='median', d_min='min', d_max='max')
)

print('Registros:', len(raw))
print('Cenários:', raw['scenario'].nunique())
print(dimension_summary.to_string())

In [ ]:
fig, axes = plt.subplots(
    3, 3,
    figsize=(FIGURE_WIDTH_CM * CM_TO_INCH, FIGURE_HEIGHT_CM * CM_TO_INCH),
)
fig.subplots_adjust(left=0.10, right=0.985, top=0.965, bottom=0.17, wspace=0.21, hspace=0.28)

for row, m in enumerate(M_VALUES):
    for column, level in enumerate(CORRELATION_LEVELS):
        scenario = f'm{m}_{level}'
        panel = summary[summary['scenario'] == scenario].sort_values('singular_index')
        x = panel['singular_index'].to_numpy(dtype=int)
        observed = panel['observed_median'].to_numpy(dtype=float)
        threshold = panel['threshold_median'].to_numpy(dtype=float)
        ax = axes[row, column]

        ax.fill_between(
            x, panel['observed_q1'], panel['observed_q3'],
            color=OBSERVED_COLOR, alpha=0.14, linewidth=0,
        )
        ax.fill_between(
            x, panel['threshold_q1'], panel['threshold_q3'],
            color=THRESHOLD_COLOR, alpha=0.12, linewidth=0,
        )
        ax.plot(
            x, observed, color=OBSERVED_COLOR, marker='o', markersize=4.2,
            linewidth=1.6, markeredgecolor='white', markeredgewidth=0.45,
        )
        ax.plot(
            x, threshold, color=THRESHOLD_COLOR, marker='s', markersize=3.8,
            linewidth=1.45, linestyle='--', markeredgecolor='white', markeredgewidth=0.4,
        )

        retained = observed > threshold
        ax.scatter(
            x[retained], observed[retained], s=34, facecolors='none',
            edgecolors=OBSERVED_COLOR, linewidths=0.8, zorder=5,
        )
        drow = dimension_summary.loc[scenario]
        d_median = int(round(float(drow['d_median'])))
        d_min = int(drow['d_min']); d_max = int(drow['d_max'])
        d_text = rf'$d={d_median}$' if d_min == d_max else rf'$d={d_median}$ [{d_min}–{d_max}]'
        ax.text(
            0.97, 0.94, d_text, transform=ax.transAxes, ha='right', va='top',
            fontsize=8.0, color='0.18',
        )

        ax.set_title(scenario, pad=4)
        ax.set_xlim(0.65, m - 0.65)
        ax.set_xticks(np.arange(1, m))
        if m == 12:
            ax.tick_params(axis='x', labelsize=6.4, pad=2)
        ymax = max(float(panel['observed_q3'].max()), float(panel['threshold_q3'].max()))
        ax.set_ylim(0, ymax * 1.10)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5, min_n_ticks=4))
        ax.grid(alpha=0.22, linewidth=0.55)
        for spine in ax.spines.values():
            spine.set_color('0.35')
            spine.set_linewidth(0.7)

fig.supxlabel(r'Direção espectral $r$', fontsize=10.0, y=0.095)
fig.supylabel('Valor singular', fontsize=10.0, x=0.025)
legend_handles = [
    Line2D([0], [0], color=OBSERVED_COLOR, marker='o', linewidth=1.6, markersize=4.5,
           label=r'Espectro observado $\sigma_r(\mathbf{E}_G)$'),
    Line2D([0], [0], color=THRESHOLD_COLOR, marker='s', linewidth=1.45, linestyle='--', markersize=4.2,
           label=r'Limiar $\tau_r$ (percentil 95 do Monte Carlo)'),
]
fig.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, 0.012), ncol=2, frameon=False)

png_path = OUT_DIR / f'{OUTPUT_STEM}.png'
pdf_path = OUT_DIR / f'{OUTPUT_STEM}.pdf'
fig.savefig(png_path, dpi=300)
fig.savefig(pdf_path, dpi=300)
plt.close(fig)

summary_path = OUT_DIR / f'{OUTPUT_STEM}_summary.csv'
summary.to_csv(summary_path, index=False)
dimension_path = OUT_DIR / f'{OUTPUT_STEM}_effective_dimension.csv'
dimension_summary.reset_index().to_csv(dimension_path, index=False)
metadata = {
    'source': SOURCE.relative_to(ROOT).as_posix(),
    'layout': '3x3; rows m=4,6,12; columns low,medium,high',
    'observed_curve': 'median of singular_value across 10 FULL seeds',
    'threshold_curve': 'median of noise_p95 across 10 FULL seeds',
    'bands': 'interquartile ranges across the same 10 seeds',
    'parallel_analysis': {'percentile': 95, 'monte_carlo_replicates': 2000, 'mc_seed': 777, 'mode': 'independent_legacy'},
    'retention_rule': 'retain direction r when singular_value > noise_p95; dimension constrained to at least 1',
    'dimension_annotation': 'median [minimum–maximum] retained_d across seeds; interval omitted when constant',
    'publication_size_cm': [FIGURE_WIDTH_CM, FIGURE_HEIGHT_CM],
    'png': png_path.relative_to(ROOT).as_posix(),
    'pdf': pdf_path.relative_to(ROOT).as_posix(),
}
metadata_path = OUT_DIR / f'{OUTPUT_STEM}_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')

for artifact in (png_path, pdf_path, summary_path, dimension_path, metadata_path):
    assert artifact.exists() and artifact.stat().st_size > 0
assert len(summary) == sum(m - 1 for m in M_VALUES) * len(CORRELATION_LEVELS)
print('Arquivos gerados:')
for artifact in (png_path, pdf_path, summary_path, dimension_path, metadata_path):
    print(' -', artifact.relative_to(ROOT).as_posix())

## Leitura da figura

- Uma direção contém sinal espectral acima do ruído quando a curva azul permanece acima da curva vermelha.
- O primeiro índice em que a curva observada deixa de superar o limiar marca a perda de suporte para dimensões adicionais.
- As faixas sombreadas mostram a variabilidade entre as dez sementes e permitem identificar decisões de retenção estáveis ou sensíveis à repetição.
- Os círculos azuis vazados reforçam as direções nas quais a mediana observada supera a mediana do limiar; a regra efetiva continua sendo aplicada separadamente em cada semente.